# 07 - Hopsworks Model Registry

## Objective
Register the best trained model (Ridge Regression)
in Hopsworks Model Registry for production use.

## Best Model
Ridge Regression
- Validation RMSE : 1.5177
- Test RMSE       : 4.6925
- Test R²         : 0.9868

## What Gets Registered
- Trained Ridge model (ridge_aqi_1h.pkl)
- Model metadata (metrics, description)
- Input schema (feature names)

## Why Model Registry
- Version control for models
- Load model in prediction pipeline
- Load model in Streamlit dashboard
- Track model performance over time

In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

import hopsworks
from dotenv import load_dotenv

print("Imports successful.")

Imports successful.


d:\talha\Pearls_AQI_Predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

HOPSWORKS_API_KEY = os.getenv("HOPSWORKS_API_KEY")

if not HOPSWORKS_API_KEY:
    raise ValueError(
        "HOPSWORKS_API_KEY was not found in the .env file."
    )

print("Hopsworks API key loaded successfully.")

Hopsworks API key loaded successfully.


In [4]:
import hopsworks

project = hopsworks.login(
    host="eu-west.cloud.hopsworks.ai",
    project="internship10P",
    api_key_value=HOPSWORKS_API_KEY,
    engine="python",
    cert_folder=r"D:\talha\Pearls_AQI_Predictor\hopsworks-certs"
)

print("Connected to Hopsworks successfully.")
print("Project:", project.name)

2026-08-19 08:30:18,087 INFO: Closing external client and cleaning up certificates.
2026-08-19 08:30:18,091 INFO: Connection closed.
2026-08-19 08:30:18,094 INFO: Initializing external client
2026-08-19 08:30:18,094 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-08-19 08:30:21,124 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42187
Connected to Hopsworks successfully.
Project: internship10P


In [5]:
# ── Best model information from notebook 06 ───────────────
MODEL_NAME    = "lahore_aqi_ridge_1h"
MODEL_VERSION = 1

METRICS = {
    "val_mae"  : 1.0044,
    "val_rmse" : 1.5177,
    "val_r2"   : 0.9988,
    "test_mae" : 2.8946,
    "test_rmse": 4.6925,
    "test_r2"  : 0.9868
}

DESCRIPTION = (
    "Ridge Regression model for one-hour-ahead "
    "AQI forecasting in Lahore. "
    "Trained on hourly weather and pollutant data "
    "from 2024-01-02 to 2025-10-22. "
    f"Test R²: {METRICS['test_r2']} | "
    f"Test RMSE: {METRICS['test_rmse']} AQI units."
)

FEATURE_COLUMNS = [
    "temperature_2m", "relative_humidity_2m",
    "surface_pressure", "precipitation",
    "cloud_cover", "wind_speed_10m",
    "wind_direction_10m", "pm2_5", "pm10",
    "carbon_monoxide", "nitrogen_dioxide",
    "sulphur_dioxide", "ozone", "us_aqi",
    "hour", "day", "month", "day_of_week",
    "is_weekend", "hour_sin", "hour_cos",
    "month_sin", "month_cos", "aqi_change_rate",
    "aqi_lag_1h", "aqi_lag_24h",
    "aqi_rolling_mean_6h", "aqi_rolling_std_6h",
    "pm25_rolling_mean_6h"
]

print("Model metadata defined.")
print("Model name   :", MODEL_NAME)
print("Description  :", DESCRIPTION[:80], "...")
print("Features     :", len(FEATURE_COLUMNS))

Model metadata defined.
Model name   : lahore_aqi_ridge_1h
Description  : Ridge Regression model for one-hour-ahead AQI forecasting in Lahore. Trained on  ...
Features     : 29


In [6]:
# Hopsworks Model Registry requires a local folder
# containing all files to upload

MODEL_DIR    = Path("../models")
REGISTRY_DIR = Path("models/registry_upload")

REGISTRY_DIR.mkdir(parents=True, exist_ok=True)

# ── Copy model file ───────────────────────────────────────
import shutil

shutil.copy(
    MODEL_DIR / "ridge_aqi_1h.pkl",
    REGISTRY_DIR / "ridge_aqi_1h.pkl"
)
print("Copied: ridge_aqi_1h.pkl")

# ── Save feature columns list ─────────────────────────────
with open(REGISTRY_DIR / "feature_columns.json", "w") as f:
    json.dump(FEATURE_COLUMNS, f, indent=2)

print("Saved: feature_columns.json")

# ── Save metrics ──────────────────────────────────────────
with open(REGISTRY_DIR / "metrics.json", "w") as f:
    json.dump(METRICS, f, indent=2)

print("Saved: metrics.json")

# ── Save model info ───────────────────────────────────────
model_info = {
    "model_name"      : MODEL_NAME,
    "model_type"      : "Ridge Regression",
    "target"          : "AQI 1 hour ahead",
    "city"            : "Lahore",
    "train_start"     : "2024-01-02",
    "train_end"       : "2025-10-22",
    "val_start"       : "2025-10-22",
    "val_end"         : "2026-03-12",
    "test_start"      : "2026-03-12",
    "test_end"        : "2026-07-31",
    "n_features"      : len(FEATURE_COLUMNS),
    "metrics"         : METRICS,
    "feature_columns" : FEATURE_COLUMNS
}

with open(REGISTRY_DIR / "model_info.json", "w") as f:
    json.dump(model_info, f, indent=2)

print("Saved: model_info.json")

# ── Verify files ──────────────────────────────────────────
print("\nFiles ready for upload:")
for f in sorted(REGISTRY_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:35s}  {size_kb:.1f} KB")

Copied: ridge_aqi_1h.pkl
Saved: feature_columns.json
Saved: metrics.json
Saved: model_info.json

Files ready for upload:
  feature_columns.json                 0.5 KB
  metrics.json                         0.1 KB
  model_info.json                      1.1 KB
  ridge_aqi_1h.pkl                     1.4 KB


In [9]:
print("Connecting to Hopsworks Model Registry...")

# ── Get model registry ────────────────────────────────────
mr = project.get_model_registry()

print("Model Registry connected.")

# ── Create a real input example with actual values ────────
# One row of realistic Lahore AQI feature values
input_example = pd.DataFrame([{
    "temperature_2m"      : 18.5,
    "relative_humidity_2m": 65.0,
    "surface_pressure"    : 1013.0,
    "precipitation"       : 0.0,
    "cloud_cover"         : 25.0,
    "wind_speed_10m"      : 5.5,
    "wind_direction_10m"  : 180.0,
    "pm2_5"               : 85.0,
    "pm10"                : 120.0,
    "carbon_monoxide"     : 800.0,
    "nitrogen_dioxide"    : 45.0,
    "sulphur_dioxide"     : 12.0,
    "ozone"               : 60.0,
    "us_aqi"              : 165.0,
    "hour"                : 10,
    "day"                 : 15,
    "month"               : 6,
    "day_of_week"         : 2,
    "is_weekend"          : 0,
    "hour_sin"            : 0.866,
    "hour_cos"            : 0.500,
    "month_sin"           : 0.866,
    "month_cos"           : 0.500,
    "aqi_change_rate"     : -0.5,
    "aqi_lag_1h"          : 166.0,
    "aqi_lag_24h"         : 160.0,
    "aqi_rolling_mean_6h" : 163.0,
    "aqi_rolling_std_6h"  : 3.5,
    "pm25_rolling_mean_6h": 83.0
}])

print("Input example created.")
print("Shape:", input_example.shape)
print("Columns:", len(input_example.columns))

# ── Verify input example is not empty ────────────────────
assert len(input_example) > 0, "Input example must not be empty"
assert len(input_example.columns) == len(FEATURE_COLUMNS), (
    f"Column count mismatch: "
    f"{len(input_example.columns)} vs {len(FEATURE_COLUMNS)}"
)

print("Input example validated.")

# ── Create model entry in registry ───────────────────────
ridge_model_entry = mr.sklearn.create_model(
    name=MODEL_NAME,
    version=MODEL_VERSION,
    metrics=METRICS,
    description=DESCRIPTION,
    input_example=input_example,
)

print("Model entry created successfully.")
print("Name   :", MODEL_NAME)
print("Version:", MODEL_VERSION)

Connecting to Hopsworks Model Registry...
Model Registry connected.
Input example created.
Shape: (1, 29)
Columns: 29
Input example validated.
Model entry created successfully.
Name   : lahore_aqi_ridge_1h
Version: 1


In [10]:
print("Uploading model files to Hopsworks...")

ridge_model_entry.save(str(REGISTRY_DIR))

print("Upload complete.")
print("Model registered successfully in Hopsworks.")

Uploading model files to Hopsworks...


Uploading model files (0 dirs, 0 files):  17%|█▋        | 1/6 [00:01<00:07,  1.47s/it]

Moving model files from 'models\registry_upload' to the model registry... This is the default behavior. Set keep_original_files=True to copy files instead.


Uploading d:\talha\Pearls_AQI_Predictor\notebooks\models\registry_upload/feature_columns.json: 100.000%|██████████| 538/538 elapsed<00:01 remaining<00:00
Uploading d:\talha\Pearls_AQI_Predictor\notebooks\models\registry_upload/metrics.json: 100.000%|██████████| 138/138 elapsed<00:01 remaining<00:00
Uploading d:\talha\Pearls_AQI_Predictor\notebooks\models\registry_upload/model_info.json: 100.000%|██████████| 1125/1125 elapsed<00:01 remaining<00:00
Uploading d:\talha\Pearls_AQI_Predictor\notebooks\models\registry_upload/ridge_aqi_1h.pkl: 100.000%|██████████| 1457/1457 elapsed<00:01 remaining<00:00
Uploading C:\Users\mtalh\AppData\Local\Temp\tmpz97gkgt8\input_example.json: 100.000%|██████████| 177/177 elapsed<00:01 remaining<00:00
Model export complete: 100%|██████████| 6/6 [00:15<00:00,  2.61s/it]                   

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/42187/models/lahore_aqi_ridge_1h/1
Upload complete.
Model registered successfully in Hopsworks.


In [11]:
print("Verifying model registration...")

# Retrieve the model back from registry
retrieved_model = mr.get_model(
    name=MODEL_NAME,
    version=MODEL_VERSION
)

print("=" * 55)
print("MODEL REGISTRY VERIFICATION")
print("=" * 55)
print("Name       :", retrieved_model.name)
print("Version    :", retrieved_model.version)
print("Description:", retrieved_model.description[:60], "...")
print()
print("Metrics:")
for key, value in retrieved_model.training_metrics.items():
    print(f"  {key:15s}: {value}")
print("=" * 55)
print("Model successfully registered in Hopsworks.")

Verifying model registration...
MODEL REGISTRY VERIFICATION
Name       : lahore_aqi_ridge_1h
Version    : 1
Description: Ridge Regression model for one-hour-ahead AQI forecasting in ...

Metrics:
  val_mae        : 1.0044
  val_rmse       : 1.5177
  test_r2        : 0.9868
  val_r2         : 0.9988
  test_rmse      : 4.6925
  test_mae       : 2.8946
Model successfully registered in Hopsworks.


In [13]:
# Simulate what the prediction pipeline will do
# Download and load the model from Hopsworks

print("Testing model download from registry...")

DOWNLOAD_DIR = Path("../models/registry_download")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Download model files
model_path = retrieved_model.download(
    local_path=str(DOWNLOAD_DIR)
)

print("Downloaded to:", model_path)

# Load the model
loaded_model = joblib.load(
    Path(model_path) / "ridge_aqi_1h.pkl"
)

print("Model loaded successfully.")
print("Model type:", type(loaded_model))

Testing model download from registry...


Downloading: 100.000%|██████████| 1457/1457 elapsed<00:00 remaining<?


Downloading: 100.000%|██████████| 1125/1125 elapsed<00:00 remaining<?


Downloading: 100.000%|██████████| 138/138 elapsed<00:00 remaining<00:00


Downloading: 100.000%|██████████| 538/538 elapsed<00:00 remaining<?

Downloaded to: ..\models\registry_downloads)... DONE
Model loaded successfully.
Model type: <class 'sklearn.linear_model._ridge.Ridge'>


In [14]:
# Make a test prediction with dummy data
# to confirm the model works after download

print("Testing prediction with loaded model...")

dummy_input = pd.DataFrame(
    np.zeros((1, len(FEATURE_COLUMNS))),
    columns=FEATURE_COLUMNS
)

test_prediction = loaded_model.predict(dummy_input)

print("Test prediction successful.")
print("Predicted AQI:", test_prediction[0])
print()
print("Model is ready for production use.")
print("It can be loaded in:")
print("  - Prediction pipeline")
print("  - Flask API")
print("  - Streamlit dashboard")

Testing prediction with loaded model...
Test prediction successful.
Predicted AQI: 47.90920294888352

Model is ready for production use.
It can be loaded in:
  - Prediction pipeline
  - Flask API
  - Streamlit dashboard


In [15]:
print()
print("=" * 60)
print("NOTEBOOK 07 COMPLETE — MODEL REGISTRY SUMMARY")
print("=" * 60)
print()
print("Model registered:")
print(f"  Name      : {MODEL_NAME}")
print(f"  Version   : {MODEL_VERSION}")
print(f"  Type      : Ridge Regression")
print()
print("Performance:")
print(f"  Val RMSE  : {METRICS['val_rmse']}")
print(f"  Val R²    : {METRICS['val_r2']}")
print(f"  Test RMSE : {METRICS['test_rmse']}")
print(f"  Test R²   : {METRICS['test_r2']}")
print()
print("Hopsworks Location:")
print(f"  Project   : internship10P")
print(f"  Registry  : Model Registry")
print(f"  Entry     : {MODEL_NAME} v{MODEL_VERSION}")
print()
print("Next Step:")
print("  08_shap_explainability.ipynb")
print("  SHAP analysis on best model (Ridge)")
print("=" * 60)


NOTEBOOK 07 COMPLETE — MODEL REGISTRY SUMMARY

Model registered:
  Name      : lahore_aqi_ridge_1h
  Version   : 1
  Type      : Ridge Regression

Performance:
  Val RMSE  : 1.5177
  Val R²    : 0.9988
  Test RMSE : 4.6925
  Test R²   : 0.9868

Hopsworks Location:
  Project   : internship10P
  Registry  : Model Registry
  Entry     : lahore_aqi_ridge_1h v1

Next Step:
  08_shap_explainability.ipynb
  SHAP analysis on best model (Ridge)
